# Capstone – Study Tip Generator
Run the cells **in order**, one at a time. Paste each cell's output into your Word file.

## Step 0 – Common Setup

In [1]:
from pathlib import Path
import random
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out = Path("lab_outputs")
out.mkdir(exist_ok=True)
print("Device:", device)

Device: cuda


## Step 0b – Install libraries
No output is normal (`-q` = quiet).

In [3]:
%pip install -q "transformers>=4.51,<5" pandas streamlit

## Step 1 – Prepare Data (training pairs and held-out topics)

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

train_pairs = [
    ("graphs", "Sketch a graph, then mark a path through three vertices."),
    ("Python loops", "Trace a loop by writing the counter after each step."),
    ("fractions", "Draw equal parts and label the fraction you shaded."),
    ("sets", "Draw a Venn diagram and label the intersection."),
    ("sorting", "Sort five values by hand and record each swap."),
    ("probability", "List the outcomes of two coin tosses before counting."),
    ("queues", "Draw a line of people and show who joins and leaves first."),
    ("linked lists", "Draw boxes joined by arrows and follow each arrow in order."),
    ("variables", "Write a value in a labelled box and change it step by step."),
    ("functions", "Write one input and the output it should return, then code it."),
    ("truth tables", "List every true and false combination before filling the result."),
    ("ratios", "Write the ratio as a fraction and check it with a small example."),
]
held_out_topics = ["recursion", "matrices", "binary search", "stacks", "SQL joins",
                   "hash tables", "derivatives", "time complexity", "pointers",
                   "normalization"]

overlap = {t for t, _ in train_pairs} & set(held_out_topics)
print("Train:", len(train_pairs), "Held-out:", len(held_out_topics), "Overlap:", overlap)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train: 12 Held-out: 10 Overlap: set()


## Step 2 – Adapt the Model (train the last Transformer block)

In [5]:
base = AutoModelForCausalLM.from_pretrained(model_id).to(device).eval()
tuned = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tuned.requires_grad_(False)
tuned.transformer.h[-1].requires_grad_(True)
opt = torch.optim.AdamW(tuned.transformer.h[-1].parameters(), lr=2e-5)

EPOCHS = 5
for epoch in range(EPOCHS):
    tuned.train()
    random.shuffle(train_pairs)
    total = 0.0
    for topic, answer in train_pairs:
        prefix = tokenizer.encode(f"Topic: {topic}\nStudy tip:")
        target = tokenizer.encode(" " + answer) + [tokenizer.eos_token_id]
        ids = torch.tensor([prefix + target], device=device)
        labels = torch.tensor([[-100] * len(prefix) + target], device=device)
        opt.zero_grad()
        loss = tuned(input_ids=ids, labels=labels).loss
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"Epoch {epoch + 1}: mean loss {total / len(train_pairs):.3f}")
tuned.eval()
tuned.save_pretrained(out / "study_tip_gpt2")
tokenizer.save_pretrained(out / "study_tip_gpt2")
print("Saved to", out / "study_tip_gpt2")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1: mean loss 5.040
Epoch 2: mean loss 4.811
Epoch 3: mean loss 4.814
Epoch 4: mean loss 4.662
Epoch 5: mean loss 4.801
Saved to lab_outputs/study_tip_gpt2


## Step 3 – Evaluate Base vs Adapted on Held-out Topics

In [6]:
import pandas as pd

def tip(net, topic):
    prompt = f"Topic: {topic}\nStudy tip:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        tokens = net.generate(**inputs, do_sample=False, max_new_tokens=45,
                              pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(tokens[0, inputs.input_ids.size(1):],
                            skip_special_tokens=True).strip()

rows = []
for topic in held_out_topics:
    rows.append({"topic": topic, "model": "base", "output": tip(base, topic)})
    rows.append({"topic": topic, "model": "adapted", "output": tip(tuned, topic)})
df = pd.DataFrame(rows)
df["relevance"] = ""; df["readability"] = ""; df["usefulness"] = ""
df.to_csv(out / "evaluation_raw_outputs.csv", index=False)
for r in rows:
    print(r["topic"], "|", r["model"], "|", r["output"], "\n")

recursion | base | Use the recursion keyword to define a new type of recursion.

Example:

class Recursion { public: Recursion(int i) { return i; } public: Recursion(int i) 

recursion | adapted | Use the recursion keyword to define a new recursion.

Example:

import recursion from ' recursion ' ; import recursion.recursion ; import recursion.recursion.recursion.rec 

matrices | base | Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as 

matrices | adapted | Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as 

binary search | base | Use the search box to find the search term you want to search for.

Study tip: Use the search box to find the search term you want to search for.

Study tip: Use the search box to 

binary se

## Step 4 – Score Summary (optional)
Only after you type your 0–2 scores into `lab_outputs/evaluation_raw_outputs.csv` (open it in the Files panel, edit, save).

In [7]:
scores = pd.read_csv(out / "evaluation_raw_outputs.csv")
scores["total"] = scores[["relevance", "readability", "usefulness"]].sum(axis=1)
print(scores.groupby("model")["total"].mean().round(2))

model
adapted    0.0
base       0.0
Name: total, dtype: float64


## Step 5 – Demonstration Interface (write app.py)
The first line must stay `%%writefile app.py`.

In [8]:
%%writefile app.py
from pathlib import Path
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

@st.cache_resource
def load_model(folder):
    tok = AutoTokenizer.from_pretrained(folder)
    net = AutoModelForCausalLM.from_pretrained(folder).eval()
    return tok, net

st.title("Study Tip Generator")
st.caption("AI-generated study tips for instructor review.")
folder = Path("lab_outputs/study_tip_gpt2")
if not (folder / "config.json").exists():
    st.error("Run the fine-tuning step and place the exported model beside this app.")
    st.stop()
topic = st.text_input("Enter a study topic", max_chars=120)
if st.button("Generate"):
    if not topic.strip():
        st.warning("Enter a study topic first.")
        st.stop()
    tokenizer, model = load_model(str(folder))
    inputs = tokenizer("Topic: " + topic.strip() + "\nStudy tip:",
                       return_tensors="pt")
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=45,
                                pad_token_id=tokenizer.eos_token_id)
    tip = tokenizer.decode(output[0, inputs.input_ids.shape[1]:],
                           skip_special_tokens=True).strip()
    st.write(tip or "No text was generated. Try a more specific topic.")

Writing app.py


## Step 6 – Launch the App (public link)
Keep this cell running while you take the screenshots. Open the `trycloudflare.com` link.

In [9]:
import subprocess, sys, time, re

subprocess.run("pkill -f 'streamlit run' || true; pkill -f cloudflared || true", shell=True)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True)

subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                 stdout=open("streamlit_logs.txt", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=open("cf_logs.txt", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cf_logs.txt").read())
    if m:
        url = m.group(0)
        break
print("APP LINK:", url or "not created - run: !cat cf_logs.txt")

APP LINK: https://chuck-charged-then-headphones.trycloudflare.com
